In [1]:
import sys
import logging
from pathlib import Path
from typing import List, Mapping, Sequence, Union, Dict
import pandas as pd
import os

import pandas as pd
import dycomutils as common_utils

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

os.chdir(repo_root)
print(repo_root)
sys.path.append(str(repo_root))

from src.utils.graph_manager import GraphManager
from src.utils.utils import load_config, format_dataframe_to_string, regex_add_strings
from src.config.experiment import ExperimentConfig
from src.experiment.ground_truth import GT, SPARQLTemplate, GTAnswer
from evaluations.test_questions.templates import build_gt_from_template

CONFIG_PATH = "evaluations/calibration/config.yaml"
logging.info(f"Loading config: {CONFIG_PATH}")
lconfig = load_config(CONFIG_PATH)
config = ExperimentConfig.model_validate(lconfig)
config


/home/desild/work/research/LLM-Workflow-Explorer


ExperimentConfig(application=ApplicationInfo(description='The application that utilizes a functional approach to support unittesting of LLMs on user designed prompts. This application contains functions generated by LLMs and functions that utilizes LLM generated output within the system. In this experiments are represented as provone:Executions of provone:Programs (a step of the experiment pipeline) while some of these provone:Programs are generated by LLM as Generative_Tasks. dc:identifier represents the unique experiment id.'), file_paths=InputFiles(schema_loc='schema/schemaV2.json', execution_kg_loc='usecases/chatbs/data/1_sample_graph/chatbs_sample.ttl', metadata_loc='usecases/chatbs/data/1_sample_graph/chatbs_sample_metadata.json'), explorer_config=ExplorerConfig(kg_name='ChatBS-NexGen', ontology_triples_path='schema/extracted_ontology_triples.csv', parallel=True, temp_folder='evaluations/calibration/exeprog_creation/tmp/programs', use_cache=False, explorer_metadata_loc='evaluatio

In [2]:
graph_manager = GraphManager(
    config.ttl,
    config.file_paths.execution_kg_loc,
)

gt_list: List[GT] = []


In [3]:
# Question 1
question1 = """
How many unique \"experiment execution\" are there in this?
"""
question_sparql1 = """
SELECT (count(distinct ?ids) AS ?obj_count)
WHERE {
     ?obj a provone:Execution ;
          dcterms:identifier ?ids .
}
"""

entities1 = graph_manager.query(question_sparql1)
entities1


,obj_count
0,1


In [4]:
answer1 = "The answer to the question is 1 unique executions."

gt_list.append(
    GT(
        question=question1,
        answer=answer1,
        sparql_querys=[
            SPARQLTemplate(
                template=question_sparql1,
                description="This SPARQL query counts the number of unique executions by counting distinct identifiers in the provone:Execution class.",
            )
        ],
        entities=entities1.to_dict(orient="records"),
    )
)


In [7]:
# Question 3
question3 = """
In what places do we utilize AI in this workflow?
"""
question_sparql3 = """
SELECT distinct ?obj ?desc
WHERE {
     ?obj a workflow:Generative_Task .
     ?obj dc:description ?desc .
}
"""

entities3 = graph_manager.query(question_sparql3, resolve_curie=True)
entities3


,obj,desc
0,ChatBS-NexGen:Generative_Task-id_2026032423045...,Generates the answer using gpt-4o@en^^<xsd:st...
1,ChatBS-NexGen:Generative_Task-id_2026032423050...,Extracts the information using gpt-4o@en^^<xs...
2,ChatBS-NexGen:Generative_Task-information_extr...,Generates the Information Extractor Function u...
3,ChatBS-NexGen:Generative_Task-query_result_pos...,Generates the post processing Function using L...
4,ChatBS-NexGen:Generative_Task-system_prompt_ge...,Generates the System Prompt Template using Lar...


In [8]:
answer3 = """
The ChatBS System utilizes LLMs to generate the following functions

1. Information extraction function
2. System Prompt template function
3. Sparql result post processing function

and further it's used within functions to
1. Generate the LLM outputs,
2. Used to extract key information from the system.
"""

gt_list.append(
    GT(
        question=question3,
        answer=answer3,
        sparql_querys=[SPARQLTemplate(template=question_sparql3, description="")],
        entities=entities3.to_dict(orient="records"),
    )
)


In [9]:
# Question 4
question4 = """
What are the extracted KG entities for the execution with id 1_1
"""
question_sparql4 = """
SELECT DISTINCT ?member ?prop ?value
WHERE {
     ?obj dc:description ?desc .
     FILTER(REGEX(?desc, "SPARQL queries to extract information", "i"))

     ?obj a provone:Program .
     ?exe prov:qualifiedAssociation/prov:hadPlan ?obj .
     ?exe dcterms:identifier ?id .
     FILTER(REGEX(?id, "1_1", "i"))

     ?data prov:wasGeneratedBy ?exe .
     ?data a ?class .

     {
         ?data a provone:Collection .
         ?data provone:hadMember ?member .
         ?member ?prop ?value .
     }
     UNION
     {
         ?data a provone:Data .
         BIND(?data AS ?member)
         ?data ?prop ?value .
     }
}
"""

entities4 = graph_manager.query(question_sparql4, resolve_curie=True)
entities4


,member,prop,value
0,ChatBS-NexGen:Data-id_20260324230500_890-kg_info,DFColumn:sugarG,4@en^^<xsd:string>
1,ChatBS-NexGen:Data-id_20260324230500_890-kg_info,DFColumn:ingredientName,Yogurt@en^^<xsd:string>
2,ChatBS-NexGen:Data-id_20260324230500_890-kg_info,DFColumn:ingredientCat,Desserts@en^^<xsd:string>
3,ChatBS-NexGen:Data-id_20260324230500_890-kg_info,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
4,ChatBS-NexGen:Data-id_20260324230500_396-kg_info,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
...,...,...,...
71,ChatBS-NexGen:Data-id_20260324230502_542-kg_info,DFColumn:ingredientCat,NA@en^^<xsd:string>
72,ChatBS-NexGen:Data-id_20260324230502_962-kg_info,DFColumn:ingredientCat,American cuisine@en^^<xsd:string>
73,ChatBS-NexGen:Data-id_20260324230502_962-kg_info,DFColumn:ingredientName,Cottage cheese@en^^<xsd:string>
74,ChatBS-NexGen:Data-id_20260324230502_962-kg_info,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data


In [10]:
pentities4 = set(
    entities4.loc[entities4["prop"] == "DFColumn:ingredientName", "value"]
    .apply(lambda x: x.split("@")[0])
    .tolist()
) - {"", "NA"}

answer4 = """
The retrieved entities from the knowledge graph includes the following ingredients,

1. Yogurt
2. Berry
3. Cheese
4. Carrot
5. Turkey
6. Bell pepper
7. Hummus
8. Almond
9. Avocado
10. Pumpkin seed
11. Water
12. Pineapple
13. Cottage cheese
"""

gt_list.append(
    GT(
        question=question4,
        answer=answer4,
        sparql_querys=[SPARQLTemplate(template=question_sparql4, description="")],
        entities=list(pentities4),
    )
)


In [11]:
# Question 5
question5 = """
why did some ingredients have missing values for `sugarG` and `ingredientCat` columns in the final output of the workflow?
"""
question_sparql5 = """
SELECT DISTINCT ?member ?prop ?value
WHERE {
     ?obj dc:description ?desc .
     FILTER(REGEX(?desc, "SPARQL queries to extract information", "i"))

     ?obj a provone:Program .
     ?exe prov:qualifiedAssociation/prov:hadPlan ?obj .
     ?exe dcterms:identifier ?id .
     FILTER(REGEX(?id, "1_1", "i"))

     ?data prov:wasGeneratedBy ?exe .
     ?data a ?class .

     {
         ?data a provone:Collection .
         ?data provone:hadMember ?member .
         ?member ?prop ?value .
     }
     UNION
     {
         ?data a provone:Data .
         BIND(?data AS ?member)
         ?data ?prop ?value .
     }
}

order by ?member ?prop
"""

entities5 = graph_manager.query(question_sparql5, resolve_curie=True)
entities5


,member,prop,value
0,ChatBS-NexGen:Data-id_20260324230500_287-kg_info,DFColumn:ingredientCat,@en^^<xsd:string>
1,ChatBS-NexGen:Data-id_20260324230500_287-kg_info,DFColumn:ingredientName,@en^^<xsd:string>
2,ChatBS-NexGen:Data-id_20260324230500_287-kg_info,DFColumn:sugarG,@en^^<xsd:string>
3,ChatBS-NexGen:Data-id_20260324230500_287-kg_info,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
4,ChatBS-NexGen:Data-id_20260324230500_396-kg_info,DFColumn:ingredientCat,Berries@en^^<xsd:string>
...,...,...,...
71,ChatBS-NexGen:Data-id_20260324230502_910-kg_info,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,provone:Data
72,ChatBS-NexGen:Data-id_20260324230502_962-kg_info,DFColumn:ingredientCat,American cuisine@en^^<xsd:string>
73,ChatBS-NexGen:Data-id_20260324230502_962-kg_info,DFColumn:ingredientName,Cottage cheese@en^^<xsd:string>
74,ChatBS-NexGen:Data-id_20260324230502_962-kg_info,DFColumn:sugarG,NA@en^^<xsd:string>


In [12]:
na_entities = entities5.loc[
    entities5["value"].isin(["@en^^<xsd:string>", "NA@en^^<xsd:string>"]),
    "member",
].unique().tolist()
pentities5 = set(
    entities5.loc[
        entities5["member"].isin(na_entities)
        & (entities5["prop"] == "DFColumn:ingredientName"),
        "value",
    ]
    .apply(lambda x: x.split("@")[0])
    .tolist()
) - {"", "NA"}

answer5 = """
Due to the fact that some information is not available in the knowledge graph for ingredients such as
'Turkey', 'Hummus', 'Cheese', 'Cottage cheese', 'Berry', 'Water' the sugarG and ingredientCat is not available.
"""

gt_list.append(
    GT(
        question=question5,
        answer=answer5,
        sparql_querys=[SPARQLTemplate(template=question_sparql5, description="")],
        entities=list(pentities5),
    )
)


In [ ]:
# Question MULT 1


template_spec = [{
    "func_desc":"sparql query post processing"
}, {
    "func_desc": "system prompt template"
}, {
    "func_desc": "information extraction"
}]

sparql_spec = [{
    "func_desc_term":"Post-processes the query results"
},{
    "func_desc_term":"Generates a system prompt"
},{
    "func_desc_term":"Extracts entities or relevant information"
},]

question_template = """
what are the instructions used by the LLM to generate the {func_desc} function step in the pipeline?
"""
sparql_template = """
SELECT distinct ?obj ?llm ?inpd
WHERE {
     ?obj dc:description ?desc .
     ?obj a ?class .
     FILTER(REGEX(?desc, "{func_desc_term}","i"))

     ?llm_out sio:SIO_000202 ?obj .
     ?llm_out sio:SIO_000232 ?llm .
     ?llm sio:SIO_000230 ?inp .
     ?inp prov:value ?inpd
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    We utilize the following input instructions to generate the function that is used to {func_desc} from the previous step.

    {answer_inst}
    """
    
    _temp_ent = entities[['obj', 'inpd']]
    _temp_ent = format_dataframe_to_string(
        _temp_ent,
        header_map={
            "obj": "Funtion",
            "inpd": "Instruction for function generation"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= entities.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

gt_list.extend(gts)

ic| rquestion: '''
                what are the instructions used by the LLM to generate the sparql query post processing function step in the pipeline?
                '''
ic| entities:                                                  obj  \
              0  http://testwebsite/testProgram#query_result_po...   
              1  http://testwebsite/testProgram#query_result_po...   
              
                                                               llm  \
              0  http://testwebsite/testProgram#LLM-query_resul...   
              1  http://testwebsite/testProgram#LLM-query_resul...   
              
                                                              inpd  
              0  Concat all the rows into single row at each co...  
              1  If there are duplicate ingredients then select...  
ic| answer: GTAnswer(answer_nlp="
                We utilize the following input instructions to generate the function that is used to sparql query post processing from t

In [14]:
gts

[GT(id='05df54ea-6b08-4c14-99cf-f209df262b95', question='\nwhat are the instructions used by the LLM to generate the sparql query post processing function step in the pipeline?\n', answer="\n    We utilize the following input instructions to generate the function that is used to sparql query post processing from the previous step.\n\n    Funtion | Instruction for function generation\nhttp://testwebsite/testProgram#query_result_post_processor | Concat all the rows into single row at each column by '|' concatenation marker.@en^^<xsd:string>\nhttp://testwebsite/testProgram#query_result_post_processor | If there are duplicate ingredients then select row with most information on `sugarG` and `ingredientCat` columns. Fill all the missing values with '0' in `sugarG` column and with '-' in 'ingredientCat` column.@en^^<xsd:string>\n    ", entities=[{'obj': 'http://testwebsite/testProgram#query_result_post_processor', 'llm': 'http://testwebsite/testProgram#LLM-query_result_post_processor', 'inpd

In [15]:
os.makedirs(os.path.dirname(config.gt.save_loc), exist_ok=True)
for i, gt in enumerate(gt_list):
    gt.id = f"gt_{i}"

common_utils.serialization.save_json(
    {gt.id: gt.model_dump() for gt in gt_list}, config.gt.save_loc
)



In [16]:
gt_list

[GT(id='gt_0', question='\nHow many unique "experiment execution" are there in this?\n', answer='The answer to the question is 1 unique executions.', entities=[{'obj_count': '1'}], sparql_querys=[SPARQLTemplate(template='\nSELECT (count(distinct ?ids) AS ?obj_count)\nWHERE {\n     ?obj a provone:Execution ;\n          dcterms:identifier ?ids .\n}\n', description='This SPARQL query counts the number of unique executions by counting distinct identifiers in the provone:Execution class.', inputs=None)]),
 GT(id='gt_1', question='\nwhat are the instructions used by the LLM to generate the sparql query post processing function step in the pipeline?\n', answer="\nWe utilize the following input instructions to generate the function that is used to post process the sparql query results from the previous step.\n\n1. Concat all the rows into single row at each column by '|' concatenation marker.@en^^<xsd:string>\n2. If there are duplicate ingredients then select row with most information on `suga